In [0]:
import org.apache.spark.sql.expressions.Aggregator
import org.apache.spark.sql.{Encoder, Encoders, SparkSession}

// Case class input
case class Input(value: Long)

// Aggregator definition
object SumOfSquares extends Aggregator[Input, Long, Long] {
  def zero: Long = 0L
  def reduce(b: Long, a: Input): Long = b + a.value * a.value
  def merge(b1: Long, b2: Long): Long = b1 + b2
  def finish(r: Long): Long = r
  def bufferEncoder: Encoder[Long] = Encoders.scalaLong
  def outputEncoder: Encoder[Long] = Encoders.scalaLong
}

// Usage
val spark = SparkSession.builder.getOrCreate()
import spark.implicits._

val ds = Seq(Input(2), Input(3), Input(4)).toDS()
val result = ds.select(SumOfSquares.toColumn)
result.show()


In [0]:
%python
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import LongType
import pandas as pd

spark = SparkSession.builder.getOrCreate()

# Sample data
df = spark.createDataFrame([(2,), (3,), (4,)], ["value"])

# Pandas UDF use Apache Arrow, convert the values into vector format
# Pandas UDF for aggregation
@F.pandas_udf(LongType())
def sum_of_squares(v: pd.Series) -> int:
    return (v ** 2).sum()

df.groupby().agg(sum_of_squares(F.col("value")).alias("sum_of_squares")).show()


In [0]:
%python
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import LongType

# sample data
df = spark.createDataFrame([(2,), (3,), (4,)], ["v"])
df.printSchema()
df.show()
# uses apache arrow to convert the data into vectorized format

@F.pandas_udf(LongType())
def sum_of_squares_pd(s: pd.Series) -> int:
    return (s ** 2).sum()

# use the DataFrame API to aggregate
agg_df = df.agg(sum_of_squares_pd(F.col("v")).alias("sos"))
agg_df.createOrReplaceTempView("sos_view")


In [0]:
%sql
SHOW TABLES in default

In [0]:
%sql
SELECT * FROM sos_view;

In [0]:
import org.apache.spark.sql.{SparkSession, Row}
import org.apache.spark.sql.expressions._
import org.apache.spark.sql.types._

// Define UDAF
class SumOfSquares extends UserDefinedAggregateFunction {
  def inputSchema: StructType = StructType(StructField("value", LongType) :: Nil)
  def bufferSchema: StructType = StructType(StructField("sum", LongType) :: Nil)
  def dataType: DataType = LongType
  def deterministic: Boolean = true

  def initialize(buffer: MutableAggregationBuffer): Unit = {
    buffer(0) = 0L
  }
  def update(buffer: MutableAggregationBuffer, input: Row): Unit = {
    if (!input.isNullAt(0)) {
      val v = input.getLong(0)
      buffer(0) = buffer.getLong(0) + v * v
    }
  }
  def merge(buffer1: MutableAggregationBuffer, buffer2: Row): Unit = {
    buffer1(0) = buffer1.getLong(0) + buffer2.getLong(0)
  }
  def evaluate(buffer: Row): Any = buffer.getLong(0)
}

// Register with Spark
val spark = SparkSession.builder().getOrCreate()
spark.udf.register("sumOfSquares", new SumOfSquares)

// Test with SQL
spark.sql("SELECT sumOfSquares(value) AS sq FROM VALUES (2), (3), (4) AS t(value)").show()


In [0]:
%sql
SELECT sumOfSquares(value) AS sq
FROM VALUES (2), (3), (4) AS t(value);
